In [0]:
%sql
CREATE VOLUME IF NOT EXISTS rapidroute.bronze.checkpoints;

In [0]:
bronze = spark.read.table("rapidroute.bronze.shipment_events")
print(f"Row count: {bronze.count()}")
bronze.printSchema()
bronze.select("shipment_id", "status", "event_timestamp", "_source_file", "_ingested_at").show(5, truncate=False)

In [0]:
# Test Schema Evolution
# Temporarily modify generate_batch to add a new field to events:
# "region": random.choice(["North", "South", "East", "West"])

from src.utils.generate_events import generate_batch

generate_batch(n_shipments=50, batch_id=4)

In [0]:
%sql
-- Re-run the Auto Loader to ingest the new data:
-- The new region column should appear in the bronze table — existing rows will have null for that column, new rows will have the value.

SELECT region, COUNT(*) FROM rapidroute.bronze.shipment_events GROUP BY region